# STIR-Net V1 — 25 pre-co-reasoning spatial proposal queries

This notebook tests one architectural question against the current high-resolution spatial-proposal implementation:

> **Should learned spatial proposal queries exist before Core Reasoning Block 1, instead of being created only after CR1 + CR2?**

The repository source is **not modified**. The experiment monkey-patches one model instance inside the notebook.

## Current architecture (Notebook 24 / repository)

```text
spatial encoder
    ↓
E3
    ↓
CR1  ↔ temporal state
    ↓
decode to E2
    ↓
CR2  ↔ temporal state
    ↓
D1 → D0
    ↓
dense center / foreground / boundary
    ↓
SpatialProposalGenerator
    ↓
proposal-specific queries
    ↓
query decoder
```

## Notebook-25 architecture

```text
spatial encoder
    ↓
spatial-only preview decode
    ↓
D0 + dense center / foreground / boundary
    ↓
SpatialProposalGenerator
    ↓
proposal-specific queries
    │
    ├──────────────┐
    ▼              │
CR1  ↔ temporal    │ proposal queries read E3
 │                 │ E3 reads proposal queries
 ▼                 │
decode to E2       │
 │                 │
 ▼                 │
CR2  ↔ temporal    │ proposal queries read E2
 │                 │ E2 reads proposal queries
 ▼                 │
D1 → D0            │
 │                 │
 ▼                 │
existing query decoder ◀──── refined proposal queries
```

The preview decode intentionally reuses the **same decoder, dense heads, proposal generator, query projections, co-reasoning attention, gates, and decoder** already present in the repository. There are **no new trainable parameters**.

This is deliberately a prototype. If it wins, the production model can later replace the duplicate preview decode with a lighter early high-resolution proposal stem.

## Main causal criteria

For source component 9 (9 GT cells), we care about:

- proposal recovery of the 9 true centers;
- number of source-9 GT cells receiving distinct matched queries;
- initial/final center separation;
- pairwise sibling mask Dice (lower is better; collapse should disappear);
- assigned coarse/native Dice (higher is better);
- source-9 internal-boundary AUC;
- global foreground Dice;
- compute / peak-memory cost.

The notebook first runs:

1. same-weight zero-step comparison;
2. one backward-step gradient gate;
3. 5-step query-bootstrap screen;
4. full step-30 → step-85 comparable run;
5. final same-weight routing ablation and Notebook-24-compatible summary.

> **Memory-fixed revision:** the early preview decoder is detached before backward. This revision specifically fixes the CUDA OOM observed in the original Notebook-25 micro backward pass.

> **Low-VRAM revision:** only source-9 proposal queries participate in the experimental pre-CR branch, and proposal→spatial feedback is disabled by default. This specifically addresses the second backward OOM while preserving the source-9 timing hypothesis.


In [ ]:
from pathlib import Path
from dataclasses import replace
from collections import defaultdict
import copy
import gc
import inspect
import json
import math
import time
import types

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
from scipy.optimize import linear_sum_assignment
from scipy.stats import rankdata

from learned.stirnet import StirNet
from learned.stirnet.debugging.acceptance.first_overfit import (
    _reduced_config,
    _repo_root,
    build_real_batch,
)
from learned.stirnet.debugging.probes.matching import (
    coarse_dice_for_matches,
    run_matching_probe,
)
from learned.stirnet.model.checkpointing import checkpoint_if_enabled
from learned.stirnet.model.coordinates import feature_grid_coordinates_um
from learned.stirnet.model.query_builder import (
    QUERY_DISCOVERY,
    QUERY_PRIMARY,
    QUERY_SPATIAL_PROPOSAL,
    QUERY_SPLIT,
    QUERY_TEMPORAL,
)
from learned.stirnet.model.types import StirNetOutput
from learned.stirnet.training.checkpoint import load_checkpoint, save_checkpoint
from learned.stirnet.training.curriculum import curriculum_stage
from learned.stirnet.training.trainer import (
    Trainer,
    model_forward_from_batch,
    move_batch_to_device,
)

SEED = 40266
SOURCE_ID = 9
AMP_DTYPE = "fp16"

# Low-VRAM pre-CR ablation:
# The all-cell scene can contain O(100) spatial proposals. Sending every one
# through an additional CR1/CR2 attention branch is not necessary to answer
# the source-9 experiment and pushes a 6 GB RTX 4050 over the edge.
PRECR_SOURCE_ONLY = SOURCE_ID
PRECR_SPATIAL_FEEDBACK = False

START_STEP = 30
FINAL_STEP = 85
MICRO_START_STEP = 50
MICRO_STEPS = 5

RUN_MICRO_SCREEN = True
RUN_FULL_COMPARABLE = True
COMPUTE_NATIVE_AT_FINAL = True
LOG_EVERY = 5

REPO_ROOT = _repo_root(Path.cwd())
DATA_DIR = (
    REPO_ROOT
    / "data"
    / "learned"
    / "stirnet"
    / "first_overfit"
    / "BlastoSPIM1_F22_030_034"
)
SPATIAL_CHECKPOINT = (
    REPO_ROOT
    / "runs"
    / "stirnet"
    / "first_overfit"
    / "12_staged_same_sample"
    / "checkpoint_spatial_dense.pt"
)
RUN_DIR = (
    REPO_ROOT
    / "runs"
    / "stirnet"
    / "first_overfit"
    / "25_pre_coreasoning_spatial_proposals"
)
RUN_DIR.mkdir(parents=True, exist_ok=True)

np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

if not torch.cuda.is_available():
    raise RuntimeError("Notebook 25 requires CUDA.")

device = torch.device("cuda")

print("Repository :", REPO_ROOT)
print("Data       :", DATA_DIR)
print("Checkpoint :", SPATIAL_CHECKPOINT)
print("Run dir    :", RUN_DIR)
print("GPU        :", torch.cuda.get_device_name(0))


## 1. Repository-contract preflight

Notebook 25 depends on the current spatial-proposal architecture. Fail early if the checked-out source predates it or its interfaces have materially changed.


In [ ]:
forward_source = inspect.getsource(StirNet.forward)

required_forward_markers = [
    "spatial_proposal_generator",
    "proposal_state",
    "self.cr1",
    "self.cr2",
    "self.query_builder",
    "self.query_decoder",
]
missing = [marker for marker in required_forward_markers if marker not in forward_source]
if missing:
    raise RuntimeError(
        "The checked-out StirNet.forward is incompatible with Notebook 25. "
        f"Missing markers: {missing}"
    )

proposal_pos = forward_source.find("self.spatial_proposal_generator")
cr2_pos = forward_source.find("self.cr2")
if proposal_pos < cr2_pos:
    raise RuntimeError(
        "The repository already appears to generate proposals before CR2; "
        "Notebook 25's assumed baseline is no longer valid."
    )

cfg_probe = _reduced_config()
if not hasattr(cfg_probe, "proposals"):
    raise RuntimeError("Current config has no proposals section.")

print("Preflight OK.")
print("Current repository order: CR2 occurs before proposal generation.")


## 2. Load the exact real all-cell scene

This is the same BlastoSPIM first-overfit scene used by the prior spatial diagnostics. Source 9 must still overlap 9 GT cells; otherwise stop rather than silently changing the test case.


In [ ]:
batch, sample = build_real_batch(DATA_DIR)
target = batch["targets"][0]

source_ids = torch.as_tensor(target["source_ids"]).long()
source_overlap = torch.as_tensor(target["source_gt_overlap"]).long()
source_row = torch.nonzero(source_ids == SOURCE_ID, as_tuple=False).flatten()
if source_row.numel() != 1:
    raise RuntimeError(
        f"Expected exactly one source row for {SOURCE_ID}; got {len(source_row)}"
    )

source9_target_indices = torch.nonzero(
    source_overlap[source_row[0]] > 0, as_tuple=False
).flatten()
source9_gt_ids = torch.as_tensor(target["ids"]).long()[source9_target_indices]

print("Spatial input shape :", tuple(batch["spatial_inputs"].shape))
print("Current components  :", int(batch["instance_ids"].numel()))
print("GT cells            :", int(torch.as_tensor(target["ids"]).numel()))
print("Source 9 GT count   :", int(source9_target_indices.numel()))
print("Source 9 GT IDs     :", source9_gt_ids.tolist())

if source9_target_indices.numel() != 9:
    raise RuntimeError(
        "Notebook 25 is specifically the 9-cell source-9 experiment; "
        f"found {source9_target_indices.numel()} compatible GT cells."
    )


## 3. Comparable reduced configuration and seed weights

The notebook starts from the exact step-30 spatial-dense checkpoint. The new proposal modules are migrated/initialized by the current repository checkpoint loader. A single migrated seed model is then copied, ensuring that post-CR and pre-CR arms begin from identical weights.


In [ ]:
def make_config():
    cfg = _reduced_config()
    cfg.proposals.enabled = True
    cfg.proposals.query_mode = "spatial_proposals"

    cfg.curriculum.enabled = True
    cfg.curriculum.spatial_dense_steps = 30
    cfg.curriculum.temporal_dense_steps = 20
    cfg.curriculum.query_bootstrap_steps = 20
    cfg.curriculum.native_bootstrap_steps = 10
    cfg.curriculum.joint_spatial_lr_scale = 0.10
    cfg.curriculum.joint_dense_lr_scale = 0.50

    # Preserve the source implementation's supervision patch.
    cfg.losses.internal_boundary = 0.25
    cfg.losses.proposal_center = 0.75
    return cfg

cfg = make_config()
seed_model = StirNet(cfg)
ckpt = load_checkpoint(
    SPATIAL_CHECKPOINT,
    seed_model,
    map_location="cpu",
    strict=True,
    migrate_history=True,
)

print("Loaded checkpoint step:", ckpt.get("step"))
print("Migration notes:", len(ckpt.get("model_migration", [])))
for note in ckpt.get("model_migration", [])[:8]:
    print(" -", note)

seed_state_cpu = copy.deepcopy(seed_model.state_dict())
del seed_model
gc.collect()


## 4. Notebook-local pre-CR proposal-query patch

### Design

1. Encode the current frame once.
2. Before CR1, run a **memory-detached spatial-only preview decode** from unmodified encoder features under `torch.no_grad()`.
3. Use the existing D0 dense heads and `SpatialProposalGenerator` to select high-resolution proposal anchors. The hard NMS/anchor selection is already non-differentiable, so the full preview decoder graph is intentionally not retained.
4. Recompute only the **sparse proposal-local embeddings** at those fixed anchors with gradients through `SpatialProposalGenerator.local_encoder`.
5. Use the existing `InstanceQueryBuilder` to convert those proposals into proposal query embeddings.
6. Feed only the proposal-query branch into CR1 and CR2:
   - proposal queries read local spatial evidence using the existing physical cross-attention;
   - spatial tokens read proposal queries using the same existing cross-attention;
   - the existing temporal branch remains unchanged;
   - no hypothesis-graph update is applied to proposal queries.
7. After CR2, build the normal query state and replace its proposal-query embeddings with the refined pre-CR proposal embeddings.
8. Run the unchanged 3-layer query decoder and native mask head.

No new `nn.Parameter` is introduced.

### Why the preview is detached

The earlier Notebook-25 draft kept both the preview D0 decoder graph and the normal post-CR D0 decoder graph alive. Backward therefore checkpoint-recomputed two full native-resolution decoder paths and exceeded a 6 GB RTX 4050. Detaching the preview is architecturally appropriate because proposal-center NMS is a hard, non-differentiable selection anyway. The sparse local encoder, query builder, CR1/CR2, main spatial path, query decoder, and native head remain trainable. The proposal score field is trained on the normal post-CR D0 path and its updated weights are used by the next iteration's early preview.


### Low-VRAM source-9 routing

The previous memory-fixed revision still routed the **entire all-cell proposal set** through an extra bidirectional CR1/CR2 branch. That is much more expensive than the causal question requires.

This revision uses:

```text
all proposals
    ↓
source-9 filter for the experimental pre-CR branch
    ↓
source-9 proposal queries read CR1 spatial features
    ↓
same queries read CR2 spatial features
    ↓
refined source-9 query identities are written back
    ↓
normal all-query decoder
```

By default, proposal queries **read** the co-reasoning spatial state, but the large spatial lattice does **not** read back from proposal queries (`PRECR_SPATIAL_FEEDBACK=False`). The original temporal↔spatial co-reasoning remains unchanged.

This still tests the central hypothesis—whether cell-specific query identities benefit from existing before CR1 and being refined by CR1/CR2—without adding a second all-proposal spatial-attention pathway across the full scene. If this source-9 experiment wins, bidirectional proposal→spatial feedback can be evaluated later on a larger GPU or with a dedicated sparse implementation.


In [ ]:
def _proposal_query_branch(
    qstate,
    proposal_state,
    dref_um,
    *,
    source_only=None,
):
    # Align proposal-query rows with proposal-state rows, then optionally keep
    # only one current source for the experimental pre-CR branch.
    all_valid = (
        (qstate.query_types == QUERY_SPATIAL_PROPOSAL)
        & (~qstate.padding_mask)
    )
    batch_grid = torch.arange(
        qstate.embeddings.shape[0],
        device=qstate.embeddings.device,
        dtype=torch.long,
    )[:, None].expand_as(all_valid)

    score_grid = qstate.embeddings.new_zeros(all_valid.shape)
    for b in range(all_valid.shape[0]):
        qrows = torch.nonzero(all_valid[b], as_tuple=False).flatten()
        prows = torch.nonzero(
            ~proposal_state.padding_mask[b], as_tuple=False
        ).flatten()
        if len(qrows) != len(prows):
            raise RuntimeError(
                f"Proposal-query alignment failed for batch {b}: "
                f"{len(qrows)} query rows vs {len(prows)} proposal rows."
            )
        if len(qrows):
            score_grid[b, qrows] = proposal_state.scores[b, prows].to(
                score_grid.dtype
            )

    selected = all_valid
    if source_only is not None:
        selected = selected & (
            qstate.source_instance_ids == int(source_only)
        )

    if not selected.any():
        raise RuntimeError(
            f"No pre-CR proposal queries selected for source={source_only}."
        )

    tokens = qstate.embeddings[selected]
    refs_um = (
        qstate.references_cellscale
        * dref_um[:, None, None]
    )[selected]
    batch_index = batch_grid[selected]
    scores = score_grid[selected, None].clamp(0.0, 1.0)

    salience = scores
    reliability = scores.clamp_min(0.05)

    return (
        tokens,
        refs_um,
        batch_index,
        salience,
        reliability,
        selected,
    )


def _replace_proposal_tokens(
    qstate,
    refined_flat_tokens,
    selected_mask,
):
    if selected_mask.shape != qstate.query_types.shape:
        raise RuntimeError("Selected pre-CR query mask shape changed.")
    if int(selected_mask.sum()) != len(refined_flat_tokens):
        raise RuntimeError(
            "Selected proposal-query count changed after CR."
        )
    embeddings = qstate.embeddings.clone()
    embeddings[selected_mask] = refined_flat_tokens.to(
        embeddings.dtype
    )
    return replace(qstate, embeddings=embeddings)



def _recompute_sparse_proposal_embeddings(
    generator,
    proposal_state,
    preview_d0,
    preview_e2,
    spatial_inputs,
    preview_dense,
    spacing_um,
    e2_spacing_um,
    dref_um,
    preview_score_logits,
):
    """
    Recompute only proposal-local embeddings with autograd enabled.

    All dense preview tensors are detached/no-grad. Therefore the only graph
    retained here is the small local_encoder graph (plus sparse sampling
    intermediates), rather than a second full native-resolution decoder graph.
    """
    max_count = proposal_state.embeddings.shape[1]
    per_batch = []

    for b in range(preview_d0.shape[0]):
        count = int((~proposal_state.padding_mask[b]).sum().item())

        if count == 0:
            local = preview_d0.new_zeros(
                (0, generator.cfg.local_dim)
            )
        else:
            refs_um = (
                proposal_state.references_cellscale[b, :count]
                * dref_um[b].clamp_min(1e-8)
            )
            local, _ = generator._local_embeddings(
                b,
                refs_um,
                preview_d0,
                preview_e2,
                spatial_inputs,
                preview_dense,
                spacing_um[b],
                e2_spacing_um[b],
                dref_um[b],
                preview_score_logits,
            )

        if count < max_count:
            padding = local.new_zeros(
                (max_count - count, generator.cfg.local_dim)
            )
            local = torch.cat([local, padding], dim=0)

        per_batch.append(local)

    embeddings = torch.stack(per_batch, dim=0)
    return replace(
        proposal_state,
        embeddings=embeddings,
    )

def proposal_aware_coreasoning(
    block,
    spatial_feature,
    spacing_um,
    temporal,
    dref_um,
    acquisition_embedding,
    proposal_tokens,
    proposal_refs_um,
    proposal_batch,
    proposal_salience,
    proposal_reliability,
    spatial_padding_mask=None,
    *,
    inject_proposals=True,
    spatial_feedback=False,
):
    # Notebook-local extension of CoReasoningBlock.forward.
    # It preserves the original temporal update, then gives proposal queries
    # a parallel spatial read and lets spatial tokens read them.
    base = spatial_feature
    s3d = block.to_model(spatial_feature)
    B, D, Z, Y, X = s3d.shape
    spatial_tokens = s3d.flatten(2).transpose(1, 2)
    pos_um = feature_grid_coordinates_um(
        (Z, Y, X), spacing_um, relative_to_center=True
    )
    normalized_spatial = block.s_norm(spatial_tokens)

    # Original temporal branch.
    if not temporal.is_empty:
        t_norm = block.t_norm1(temporal.tokens)
        t_msg = block.cross.temporal_reads_spatial(
            t_norm,
            temporal.ref_um,
            temporal.salience,
            normalized_spatial,
            pos_um,
            temporal.batch_index,
            dref_um,
            spatial_padding_mask,
            block.cfg.base_radius_dref,
            temporal.history_support,
            temporal.history_support_valid,
            temporal.history_support_dt,
            temporal.history_support_center_um,
            temporal.history_support_extent_um,
        )
        gate_in = torch.cat(
            [
                temporal.tokens,
                t_msg,
                temporal.salience,
                temporal.reliability,
            ],
            dim=-1,
        )
        t = temporal.tokens + block.t_gate(gate_in) * t_msg
        t = t + block.t_ffn(block.t_norm2(t))
        t = block.hyp_graph(t, temporal.edge_index, temporal.edge_attr)
        temporal = replace(temporal, tokens=t)

        s_msg_t = block.cross.spatial_reads_temporal(
            normalized_spatial,
            pos_um,
            temporal.tokens,
            temporal.ref_um,
            temporal.salience,
            temporal.reliability,
            temporal.batch_index,
            dref_um,
            spatial_padding_mask,
            block.cfg.base_radius_dref,
        )
        spatial_tokens = checkpoint_if_enabled(
            block._gated_spatial_update,
            spatial_tokens,
            s_msg_t,
            enabled=block.activation_checkpointing and block.training,
        )

    # New proposal-query branch. Reuses the CR attention/gating weights but
    # deliberately does not enter the temporal hypothesis graph.
    refined_proposals = proposal_tokens
    if inject_proposals and proposal_tokens.numel():
        p_norm = block.t_norm1(proposal_tokens)
        p_msg = block.cross.temporal_reads_spatial(
            p_norm,
            proposal_refs_um,
            proposal_salience,
            normalized_spatial,
            pos_um,
            proposal_batch,
            dref_um,
            spatial_padding_mask,
            block.cfg.base_radius_dref,
            None,
            None,
            None,
            None,
            None,
        )
        p_gate_in = torch.cat(
            [
                proposal_tokens,
                p_msg,
                proposal_salience,
                proposal_reliability,
            ],
            dim=-1,
        )
        refined_proposals = (
            proposal_tokens
            + block.t_gate(p_gate_in) * p_msg
        )
        refined_proposals = (
            refined_proposals
            + block.t_ffn(block.t_norm2(refined_proposals))
        )

        if spatial_feedback:
            s_msg_p = block.cross.spatial_reads_temporal(
                normalized_spatial,
                pos_um,
                refined_proposals,
                proposal_refs_um,
                proposal_salience,
                proposal_reliability,
                proposal_batch,
                dref_um,
                spatial_padding_mask,
                block.cfg.base_radius_dref,
            )
            spatial_tokens = checkpoint_if_enabled(
                block._gated_spatial_update,
                spatial_tokens,
                s_msg_p,
                enabled=block.activation_checkpointing and block.training,
            )

    def refine_spatial(tokens, embedding):
        return block._refine_spatial_tokens(
            tokens, embedding, (B, D, Z, Y, X)
        )

    refined = checkpoint_if_enabled(
        refine_spatial,
        spatial_tokens,
        acquisition_embedding,
        enabled=block.activation_checkpointing and block.training,
    )
    updated = base + refined
    return updated, temporal, refined_proposals


In [ ]:
def forward_precr_proposals(
    self,
    spatial_inputs,
    instance_labels,
    spacing_um,
    dref_um,
    instance_features,
    instance_ids,
    instance_batch,
    instance_centroids_um,
    graph_x,
    graph_edge_index,
    graph_edge_attr,
    tracklet_id,
    temporal_ref_um,
    temporal_status,
    hypothesis_edge_index,
    hypothesis_edge_attr,
    temporal_batch,
    spatial_padding_mask=None,
    return_debug=False,
    bypass_coreasoning=False,
    node_instance_grid=None,
    node_history_valid=None,
    history_support=None,
    history_support_valid=None,
    history_support_dt=None,
    history_support_center_um=None,
    history_support_extent_um=None,
    best_current_component_id=None,
    best_component_overlap=None,
    second_best_component_overlap=None,
    node_observed_ref_um=None,
    node_time_offset=None,
    node_ids=None,
    temporal_memory_ablation="full",
    detection_graph_ablation="full",
    return_full_temporal_attention=False,
):
    acq = self.acquisition(spacing_um, dref_um)
    pyramid = self.encoder(
        spatial_inputs, spacing_um, acq, spatial_padding_mask
    )

    temporal = self._build_temporal(
        graph_x,
        graph_edge_index,
        graph_edge_attr,
        tracklet_id,
        temporal_ref_um,
        temporal_status,
        hypothesis_edge_index,
        hypothesis_edge_attr,
        temporal_batch,
        dref_um,
        node_instance_grid,
        node_history_valid,
        history_support,
        history_support_valid,
        history_support_dt,
        history_support_center_um,
        history_support_extent_um,
        best_current_component_id,
        best_component_overlap,
        second_best_component_overlap,
        node_observed_ref_um,
        node_time_offset,
        node_ids,
        detection_graph_ablation,
    )

    # ---------------------------------------------------------------
    # EARLY SPATIAL-ONLY PREVIEW — MEMORY-DETACHED
    # ---------------------------------------------------------------
    # Hard proposal-center selection already runs through no-grad NMS.
    # Retaining a second full D0 decoder graph only wastes VRAM and caused the
    # original Notebook-25 backward OOM on a 6 GB RTX 4050.
    with torch.no_grad():
        preview_e3 = pyramid.features[3].detach()
        preview_e2 = self.decoder.decode_to_e2(
            preview_e3, pyramid, acq
        )
        preview_d1, preview_d0, _ = self.decoder.decode_from_e2(
            preview_e2, pyramid, acq
        )
        preview_dense = self.dense_heads(preview_d0)
        proposal_state, preview_proposal_score_logits = (
            self.spatial_proposal_generator(
                preview_d0,
                preview_e2,
                spatial_inputs,
                preview_dense,
                instance_labels,
                spacing_um,
                pyramid.spacings_um[2],
                dref_um,
                instance_ids,
                instance_batch,
                instance_centroids_um,
                spatial_padding_mask,
            )
        )

    # Re-enable gradients only for the sparse proposal-local representation.
    # This trains SpatialProposalGenerator.local_encoder without retaining the
    # preview decoder / native dense computation graph.
    proposal_state = _recompute_sparse_proposal_embeddings(
        self.spatial_proposal_generator,
        proposal_state,
        preview_d0.detach(),
        preview_e2.detach(),
        spatial_inputs.detach(),
        {k: v.detach() for k, v in preview_dense.items()},
        spacing_um,
        pyramid.spacings_um[2],
        dref_um,
        preview_proposal_score_logits.detach(),
    )

    # Build proposal-specific query identities BEFORE CR1.
    pre_qstate = self.query_builder(
        preview_e2,
        pyramid.spacings_um[2],
        instance_labels,
        instance_features,
        instance_ids,
        instance_batch,
        instance_centroids_um,
        dref_um,
        temporal,
        memory_ablation=temporal_memory_ablation,
        return_debug=return_debug,
        full_attention=return_full_temporal_attention,
        proposal_state=proposal_state,
        query_mode="spatial_proposals",
    )
    (
        p_tokens0,
        p_refs_um,
        p_batch,
        p_salience,
        p_reliability,
        p_selected_mask,
    ) = _proposal_query_branch(
        pre_qstate,
        proposal_state,
        dref_um,
        source_only=PRECR_SOURCE_ONLY,
    )

    # The proposal-local Linear graph retains only small sampled vectors.
    # Release the full detached preview volumes before constructing CR1/CR2.
    del preview_d1, preview_d0, preview_dense, preview_e2, preview_e3

    inject = bool(
        getattr(self, "_notebook25_inject_proposals", True)
    )
    spatial_feedback = bool(
        getattr(
            self,
            "_notebook25_spatial_feedback",
            PRECR_SPATIAL_FEEDBACK,
        )
    )

    # ---------------------------------------------------------------
    # CR1 / CR2 WITH EARLY PROPOSAL QUERY FEEDBACK
    # ---------------------------------------------------------------
    if bypass_coreasoning:
        # Keep the real spatial path trainable. The detached preview is used
        # only to choose/describe early proposals.
        e3 = pyramid.features[3]
        e2 = self.decoder.decode_to_e2(e3, pyramid, acq)
        p_tokens1 = p_tokens0
        p_tokens2 = p_tokens0
    else:
        e3, temporal, p_tokens1 = proposal_aware_coreasoning(
            self.cr1,
            pyramid.features[3],
            pyramid.spacings_um[3],
            temporal,
            dref_um,
            acq,
            p_tokens0,
            p_refs_um,
            p_batch,
            p_salience,
            p_reliability,
            pyramid.padding_masks[3] if pyramid.padding_masks else None,
            inject_proposals=inject,
            spatial_feedback=spatial_feedback,
        )

        e2 = self.decoder.decode_to_e2(e3, pyramid, acq)

        e2, temporal, p_tokens2 = proposal_aware_coreasoning(
            self.cr2,
            e2,
            pyramid.spacings_um[2],
            temporal,
            dref_um,
            acq,
            p_tokens1,
            p_refs_um,
            p_batch,
            p_salience,
            p_reliability,
            pyramid.padding_masks[2] if pyramid.padding_masks else None,
            inject_proposals=inject,
            spatial_feedback=spatial_feedback,
        )

    d1, d0, mask_features = self.decoder.decode_from_e2(
        e2, pyramid, acq
    )
    dense = self.dense_heads(d0)

    # Train the proposal score field on the normal post-CR spatial path.
    # The next optimization iteration's detached early preview uses these
    # updated score-head weights. This avoids retaining two native-resolution
    # score/decoder graphs in the same backward pass.
    dense["proposal_score_logits"] = (
        self.spatial_proposal_generator.proposal_score_logits(
            d0,
            spatial_inputs,
            dense,
        )
    )

    # Build normal temporal + discovery + proposal query layout AFTER CR2,
    # but keep proposal IDs/anchors from the PRE-CR proposal state.
    qstate = self.query_builder(
        e2,
        pyramid.spacings_um[2],
        instance_labels,
        instance_features,
        instance_ids,
        instance_batch,
        instance_centroids_um,
        dref_um,
        temporal,
        memory_ablation=temporal_memory_ablation,
        return_debug=return_debug,
        full_attention=return_full_temporal_attention,
        proposal_state=proposal_state,
        query_mode="spatial_proposals",
    )

    # Proposal embeddings entering the existing query decoder are the
    # identities refined through CR1/CR2, not freshly rebuilt post-CR.
    qstate = _replace_proposal_tokens(
        qstate,
        p_tokens2,
        p_selected_mask,
    )
    initial_query_references = qstate.references_cellscale

    qstate, dec_outputs = self.query_decoder(
        qstate,
        [e3, e2, d1],
        [
            pyramid.spacings_um[3],
            pyramid.spacings_um[2],
            pyramid.spacings_um[1],
        ],
        instance_labels,
        dref_um,
        temporal,
        memory_ablation=temporal_memory_ablation,
        return_debug=return_debug,
        full_attention=return_full_temporal_attention,
    )

    final = dec_outputs[-1]
    native_emb = self.native_mask_head(qstate.embeddings)

    debug = None
    if return_debug:
        valid_proposals = ~proposal_state.padding_mask
        debug = {
            "query_mode": "spatial_proposals_pre_cr",
            "precr_inject_proposals": inject,
            "precr_source_only": PRECR_SOURCE_ONLY,
            "precr_spatial_feedback": spatial_feedback,
            "precr_selected_query_count": torch.tensor(
                int(p_selected_mask.sum().item()),
                device=spatial_inputs.device,
            ),
            "proposal_score_logits": preview_proposal_score_logits.detach(),
            "proposal_references_cellscale": (
                proposal_state.references_cellscale.detach()
            ),
            "proposal_scores": proposal_state.scores.detach(),
            "proposal_source_instance_ids": (
                proposal_state.source_instance_ids.detach()
            ),
            "proposal_fallback_mask": proposal_state.fallback_mask.detach(),
            "proposal_padding_mask": proposal_state.padding_mask.detach(),
            "learned_proposal_count": (
                (valid_proposals & ~proposal_state.fallback_mask)
                .sum(dim=1)
                .detach()
            ),
            "fallback_proposal_count": (
                (valid_proposals & proposal_state.fallback_mask)
                .sum(dim=1)
                .detach()
            ),
            "precr_proposal_embeddings_initial_flat": p_tokens0.detach(),
            "precr_proposal_embeddings_after_cr1_flat": p_tokens1.detach(),
            "precr_proposal_embeddings_after_cr2_flat": p_tokens2.detach(),
            "precr_proposal_refs_um_flat": p_refs_um.detach(),
            "precr_proposal_batch_flat": p_batch.detach(),
            "precr_proposal_scores_flat": p_salience.detach(),
            "query_initial_references_cellscale": (
                initial_query_references.detach()
            ),
            "query_layer_references_cellscale": torch.stack(
                [layer["centers_cellscale"] for layer in dec_outputs],
                dim=0,
            ).detach(),
            "query_references_cellscale": (
                qstate.references_cellscale.detach()
            ),
        }

    return StirNetOutput(
        exist_logits=final["exist_logits"],
        centers_cellscale=final["centers_cellscale"],
        coarse_mask_logits=final["coarse_mask_logits"],
        coarse_spacing_um=final["coarse_spacing_um"],
        query_embeddings=qstate.embeddings,
        native_mask_embeddings=native_emb,
        query_types=qstate.query_types,
        query_padding_mask=qstate.padding_mask,
        source_instance_ids=qstate.source_instance_ids,
        query_initial_references_cellscale=initial_query_references,
        temporal_salience=qstate.temporal_salience,
        temporal_reliability=qstate.temporal_reliability,
        aux_outputs=dec_outputs[:-1],
        dense_outputs=dense,
        mask_features=mask_features,
        spacing_um=spacing_um,
        dref_um=dref_um,
        instance_labels=instance_labels,
        debug=debug,
        proposals=proposal_state,
    )


def bind_precr_patch(
    model,
    *,
    inject_proposals=True,
    spatial_feedback=PRECR_SPATIAL_FEEDBACK,
):
    model.forward = types.MethodType(
        forward_precr_proposals, model
    )
    model._notebook25_inject_proposals = bool(
        inject_proposals
    )
    model._notebook25_spatial_feedback = bool(
        spatial_feedback
    )
    return model


## 5. Structural metrics

The comparison is deliberately centered on decomposition, not just total loss.

AUC is measured for the learned boundary logits on source 9 using:

- positive = GT internal cell-cell boundary voxels inside source 9;
- negative = source-9 cell voxels that are not any GT boundary.

Pairwise sibling Dice is measured on matched source-9 query masks. High pairwise Dice means different queries are still collapsing to the same mask.


In [ ]:
QUERY_NAMES = {
    QUERY_PRIMARY: "primary",
    QUERY_SPLIT: "split",
    QUERY_TEMPORAL: "temporal",
    QUERY_DISCOVERY: "discovery",
    QUERY_SPATIAL_PROPOSAL: "spatial_proposal",
}


def binary_auc(scores, labels):
    scores = np.asarray(scores, np.float64)
    labels = np.asarray(labels, bool)
    n_pos = int(labels.sum())
    n_neg = int((~labels).sum())
    if n_pos == 0 or n_neg == 0:
        return np.nan
    ranks = rankdata(scores)
    return float(
        (ranks[labels].sum() - n_pos * (n_pos + 1) / 2)
        / (n_pos * n_neg)
    )


def pairwise_soft_dice(probabilities):
    if len(probabilities) < 2:
        return []
    flat = probabilities.float().flatten(1)
    values = []
    for i in range(len(flat)):
        for j in range(i + 1, len(flat)):
            a, b = flat[i], flat[j]
            values.append(
                float(
                    (
                        (2 * (a * b).sum() + 1e-6)
                        / (a.sum() + b.sum() + 1e-6)
                    ).detach().cpu()
                )
            )
    return values


def cosine_pairwise(tokens):
    if tokens is None or len(tokens) < 2:
        return []
    x = F.normalize(tokens.float(), dim=-1)
    sim = x @ x.T
    tri = torch.triu_indices(
        len(x), len(x), offset=1, device=x.device
    )
    return sim[tri[0], tri[1]].detach().cpu().tolist()


def source9_target_info(target):
    source_ids = torch.as_tensor(target["source_ids"]).long()
    overlap = torch.as_tensor(target["source_gt_overlap"]).long()
    row = torch.nonzero(
        source_ids == SOURCE_ID, as_tuple=False
    ).flatten()
    if len(row) != 1:
        raise RuntimeError("Source 9 compatibility row missing.")
    target_idx = torch.nonzero(
        overlap[row[0]] > 0, as_tuple=False
    ).flatten()
    ids = torch.as_tensor(target["ids"]).long()[target_idx]
    centers = torch.as_tensor(
        target["centers_cellscale"]
    ).float()[target_idx]
    return target_idx, ids, centers


def proposal_center_metrics(outputs, target):
    _, _, gt_centers = source9_target_info(target)
    dref = float(outputs.dref_um[0].detach().cpu())

    state = outputs.proposals
    valid = ~state.padding_mask[0]
    refs = state.references_cellscale[
        0, valid
    ].detach().float().cpu()
    src = state.source_instance_ids[
        0, valid
    ].detach().cpu()
    fallback = state.fallback_mask[
        0, valid
    ].detach().cpu()

    source9 = src == SOURCE_ID
    refs9 = refs[source9]

    result = {
        "proposal_count_total": int(valid.sum().item()),
        "proposal_count_source9": int(source9.sum().item()),
        "proposal_count_source9_learned": int(
            (source9 & ~fallback).sum().item()
        ),
    }

    if len(refs9) == 0:
        result.update(
            {
                "proposal_center_mean_error_um": np.nan,
                "proposal_center_median_error_um": np.nan,
                "proposal_recovery_0p5_dref": 0,
                "proposal_recovery_1p0_dref": 0,
            }
        )
        return result

    dist = torch.cdist(refs9, gt_centers) * dref
    rows, cols = linear_sum_assignment(dist.numpy())
    matched = dist[rows, cols].numpy()
    result.update(
        {
            "proposal_center_mean_error_um": float(
                matched.mean()
            ),
            "proposal_center_median_error_um": float(
                np.median(matched)
            ),
            "proposal_recovery_0p5_dref": int(
                np.count_nonzero(matched <= 0.5 * dref)
            ),
            "proposal_recovery_1p0_dref": int(
                np.count_nonzero(matched <= 1.0 * dref)
            ),
        }
    )
    return result


def source9_boundary_auc(outputs, target):
    boundary_logits = outputs.dense_outputs[
        "boundary_logits"
    ][0, 0]
    labels_current = outputs.instance_labels[0]

    internal = torch.as_tensor(
        target["internal_boundary"],
        device=boundary_logits.device,
    ).bool()
    all_boundary = torch.as_tensor(
        target["boundary"],
        device=boundary_logits.device,
    ).bool()

    source = labels_current == SOURCE_ID
    positive = source & internal
    negative = source & (~all_boundary)

    if not positive.any() or not negative.any():
        return np.nan, np.nan

    scores = torch.cat(
        [
            boundary_logits[positive],
            boundary_logits[negative],
        ]
    ).detach().float().cpu().numpy()
    labels = np.concatenate(
        [
            np.ones(
                int(positive.sum().item()), dtype=bool
            ),
            np.zeros(
                int(negative.sum().item()), dtype=bool
            ),
        ]
    )
    auc = binary_auc(scores, labels)
    recall = float(
        (boundary_logits[positive].sigmoid() >= 0.5)
        .float()
        .mean()
        .detach()
        .cpu()
    )
    return auc, recall


def foreground_hard_dice(outputs, target):
    pred = (
        outputs.dense_outputs["foreground_logits"][0, 0]
        >= 0
    )
    gt = torch.as_tensor(
        target["foreground"], device=pred.device
    ).bool()
    inter = (pred & gt).sum().float()
    return float(
        (
            (2 * inter + 1e-6)
            / (pred.sum() + gt.sum() + 1e-6)
        )
        .detach()
        .cpu()
    )


def source9_matching_metrics(outputs, target):
    probe = run_matching_probe(outputs, [target])
    match = probe.matches[0]
    source9_idx, source9_ids, _ = source9_target_info(
        target
    )
    source9_set = set(source9_idx.tolist())

    q_selected = []
    t_selected = []
    for q, t in zip(
        match.pred_indices.detach().cpu().tolist(),
        match.target_indices.detach().cpu().tolist(),
    ):
        if int(t) in source9_set:
            q_selected.append(int(q))
            t_selected.append(int(t))

    coarse_map = coarse_dice_for_matches(
        outputs.coarse_mask_logits[0], target, match
    )
    coarse_values = [
        coarse_map[q]
        for q in q_selected
        if q in coarse_map
    ]

    type_counts = defaultdict(int)
    for q in q_selected:
        qt = int(
            outputs.query_types[0, q].detach().cpu()
        )
        type_counts[
            QUERY_NAMES.get(qt, str(qt))
        ] += 1

    center_errors = []
    source9_id_set = set(
        int(x) for x in source9_ids.tolist()
    )
    for row in probe.rows:
        if int(row["gt_id"]) in source9_id_set:
            center_errors.append(
                float(row["center_error_um"])
            )

    selected_tensor = torch.as_tensor(
        q_selected,
        device=outputs.exist_logits.device,
        dtype=torch.long,
    )
    if len(q_selected) >= 2:
        coarse_probs = outputs.coarse_mask_logits[
            0, selected_tensor
        ].sigmoid()
        sibling = pairwise_soft_dice(coarse_probs)
    else:
        sibling = []

    proposal_q = (
        (
            outputs.query_types[0]
            == QUERY_SPATIAL_PROPOSAL
        )
        & (~outputs.query_padding_mask[0])
        & (
            outputs.source_instance_ids[0]
            == SOURCE_ID
        )
    )
    final_cos = cosine_pairwise(
        outputs.query_embeddings[0, proposal_q]
    )

    return {
        "source9_matched_gt_count": len(q_selected),
        "source9_matched_gt_ids": [
            int(torch.as_tensor(target["ids"])[t])
            for t in t_selected
        ],
        "source9_match_query_type_counts": dict(
            type_counts
        ),
        "source9_coarse_dice_mean": (
            float(np.mean(coarse_values))
            if coarse_values
            else np.nan
        ),
        "source9_coarse_dice_median": (
            float(np.median(coarse_values))
            if coarse_values
            else np.nan
        ),
        "source9_center_error_mean_um": (
            float(np.mean(center_errors))
            if center_errors
            else np.nan
        ),
        "source9_pairwise_coarse_dice_mean": (
            float(np.mean(sibling))
            if sibling
            else np.nan
        ),
        "source9_pairwise_coarse_dice_median": (
            float(np.median(sibling))
            if sibling
            else np.nan
        ),
        "source9_final_query_cosine_mean": (
            float(np.mean(final_cos))
            if final_cos
            else np.nan
        ),
        "_source9_q_selected": q_selected,
        "_source9_t_selected": t_selected,
    }


@torch.no_grad()
def source9_native_metrics(
    model,
    outputs,
    target,
    q_selected,
    t_selected,
):
    # Render one query at a time to keep laptop-GPU memory bounded.
    if not q_selected:
        return {
            "source9_native_dice_mean": np.nan,
            "source9_native_dice_median": np.nan,
            "source9_pairwise_native_dice_mean": np.nan,
            "source9_pairwise_native_dice_median": np.nan,
        }

    label_map_cpu = torch.as_tensor(
        target["label_map"]
    ).long()
    target_ids = torch.as_tensor(
        target["ids"]
    ).long()

    source9_target_idx, _, _ = source9_target_info(
        target
    )
    source9_gt_ids = target_ids[source9_target_idx]
    source9_mask = torch.zeros_like(
        label_map_cpu, dtype=torch.bool
    )
    for gt_id in source9_gt_ids.tolist():
        source9_mask |= (
            label_map_cpu == int(gt_id)
        )

    coords = torch.nonzero(
        source9_mask, as_tuple=False
    )
    lo = coords.min(0).values
    hi = coords.max(0).values + 1
    margin = torch.tensor([2, 4, 4])
    lo = torch.maximum(
        lo - margin, torch.zeros_like(lo)
    )
    hi = torch.minimum(
        hi + margin,
        torch.tensor(
            label_map_cpu.shape,
            dtype=torch.long,
        ),
    )
    sl = tuple(
        slice(int(a), int(b))
        for a, b in zip(lo, hi)
    )

    assigned = []
    cropped_probs = []

    for q, t in zip(q_selected, t_selected):
        qidx = torch.tensor(
            [q],
            device=outputs.exist_logits.device,
            dtype=torch.long,
        )
        logits = model.render_masks(
            outputs, [qidx]
        )[0][0]
        prob = logits.sigmoid()

        gt_id = int(target_ids[t])
        gt = (
            label_map_cpu == gt_id
        ).to(
            prob.device,
            non_blocking=True,
        )
        inter = (prob * gt).sum()
        dice = (
            (2 * inter + 1e-6)
            / (prob.sum() + gt.sum() + 1e-6)
        )
        assigned.append(
            float(dice.detach().cpu())
        )

        cropped_probs.append(
            prob[sl]
            .detach()
            .to(
                "cpu",
                dtype=torch.float16,
            )
        )
        del logits, prob, gt
        torch.cuda.empty_cache()

    cropped = torch.stack(
        cropped_probs
    ).float()
    sibling = pairwise_soft_dice(cropped)

    return {
        "source9_native_dice_mean": float(
            np.mean(assigned)
        ),
        "source9_native_dice_median": float(
            np.median(assigned)
        ),
        "source9_pairwise_native_dice_mean": (
            float(np.mean(sibling))
            if sibling
            else np.nan
        ),
        "source9_pairwise_native_dice_median": (
            float(np.median(sibling))
            if sibling
            else np.nan
        ),
    }


@torch.no_grad()
def evaluate_structure(
    model,
    batch_gpu,
    target,
    *,
    label,
    native=False,
    temporal_memory_ablation="full",
):
    model.eval()
    torch.cuda.reset_peak_memory_stats()
    start = time.perf_counter()
    with torch.autocast(
        "cuda", dtype=torch.float16
    ):
        outputs = model_forward_from_batch(
            model,
            batch_gpu,
            return_debug=True,
            temporal_memory_ablation=(
                temporal_memory_ablation
            ),
        )
    torch.cuda.synchronize()
    elapsed = time.perf_counter() - start
    peak_gib = (
        torch.cuda.max_memory_allocated()
        / (1024**3)
    )

    metrics = {
        "label": label,
        "forward_seconds": elapsed,
        "peak_memory_gib": peak_gib,
        "foreground_hard_dice": (
            foreground_hard_dice(
                outputs, target
            )
        ),
    }

    auc, recall = source9_boundary_auc(
        outputs, target
    )
    metrics[
        "source9_internal_boundary_auc"
    ] = auc
    metrics[
        "source9_internal_boundary_recall_0p5"
    ] = recall
    metrics.update(
        proposal_center_metrics(
            outputs, target
        )
    )

    match_metrics = source9_matching_metrics(
        outputs, target
    )
    q_selected = match_metrics.pop(
        "_source9_q_selected"
    )
    t_selected = match_metrics.pop(
        "_source9_t_selected"
    )
    metrics.update(match_metrics)

    if outputs.debug is not None:
        for key, out_name in [
            (
                "precr_proposal_embeddings_initial_flat",
                "precr_initial_query_cosine_mean",
            ),
            (
                "precr_proposal_embeddings_after_cr1_flat",
                "precr_after_cr1_query_cosine_mean",
            ),
            (
                "precr_proposal_embeddings_after_cr2_flat",
                "precr_after_cr2_query_cosine_mean",
            ),
        ]:
            tokens = outputs.debug.get(key)
            if tokens is not None:
                values = cosine_pairwise(tokens)
                metrics[out_name] = (
                    float(np.mean(values))
                    if values
                    else np.nan
                )

    if native:
        metrics.update(
            source9_native_metrics(
                model,
                outputs,
                target,
                q_selected,
                t_selected,
            )
        )

    del outputs
    torch.cuda.empty_cache()
    return metrics


## 6. Zero-step same-weight causal comparison

Both arms use exactly the same migrated step-30 weights.

- **post_cr_current**: repository architecture.
- **pre_cr_feedback**: Notebook-25 early proposals participate in CR1/CR2.

This is the cleanest test of whether moving proposal identity earlier changes the forward mechanism before any retraining.


In [ ]:
baseline_model = StirNet(cfg)
baseline_model.load_state_dict(
    seed_state_cpu, strict=True
)
baseline_model.to(device)

early_model_zero = StirNet(cfg)
early_model_zero.load_state_dict(
    seed_state_cpu, strict=True
)
bind_precr_patch(
    early_model_zero,
    inject_proposals=True,
)
early_model_zero.to(device)

batch_gpu = move_batch_to_device(
    batch, device
)

zero_rows = []
zero_rows.append(
    evaluate_structure(
        baseline_model,
        batch_gpu,
        target,
        label="post_cr_current_step30",
        native=False,
    )
)
zero_rows.append(
    evaluate_structure(
        early_model_zero,
        batch_gpu,
        target,
        label="pre_cr_feedback_step30",
        native=False,
    )
)

zero_df = pd.DataFrame(zero_rows)
display(zero_df.T)

zero_df.to_csv(
    RUN_DIR / "zero_step_comparison.csv",
    index=False,
)

del baseline_model, early_model_zero
torch.cuda.empty_cache()
gc.collect()


## 7. One backward-step gate + 5-step query-bootstrap screen

The long staged run is not the first place to discover a disconnected patch.

For the micro screen we jump a clean copy to **query_bootstrap step 50**, where instance-query losses are active, and verify gradients reach:

- `spatial_proposal_generator.local_encoder`;
- `cr1.cross`;
- `cr2.cross`;
- `query_builder.proposal_proj`;
- `query_decoder`.

Then we run only 5 optimization steps and remeasure source-9 structure.

> **Important after the previous CUDA OOM:** restart the Jupyter kernel before running this corrected notebook from the top. A failed CUDA allocation during checkpoint RNG restoration can leave the current process with fragmented/reserved VRAM even after `empty_cache()`.

In this low-VRAM revision, `PRECR_SPATIAL_FEEDBACK=False` is intentional. The gradient gate therefore checks that source-9 proposal queries train through the CR cross-attention read path; it does not require the spatial lattice to receive a second proposal-originating message.


In [ ]:
def module_grad_norm(module):
    sq = 0.0
    found = False
    for p in module.parameters():
        if p.grad is not None:
            found = True
            sq += float(
                p.grad.detach()
                .float()
                .pow(2)
                .sum()
                .cpu()
            )
    return math.sqrt(sq) if found else 0.0


micro_result = None

# Make sure evaluation leftovers are released before the first backward pass.
gc.collect()
torch.cuda.empty_cache()
print(
    "Low-VRAM pre-CR branch:",
    f"source={PRECR_SOURCE_ONLY}",
    f"spatial_feedback={PRECR_SPATIAL_FEEDBACK}",
)
print(
    "CUDA before micro backward | allocated:",
    f"{torch.cuda.memory_allocated() / 1024**3:.2f} GiB",
    "| reserved:",
    f"{torch.cuda.memory_reserved() / 1024**3:.2f} GiB",
)

if RUN_MICRO_SCREEN:
    micro_model = StirNet(cfg)
    micro_model.load_state_dict(
        seed_state_cpu, strict=True
    )
    bind_precr_patch(
        micro_model,
        inject_proposals=True,
    )

    micro_trainer = Trainer(
        micro_model,
        cfg,
        device=device,
        amp_dtype=AMP_DTYPE,
    )
    micro_trainer.global_step = (
        MICRO_START_STEP
    )
    micro_trainer.curriculum_stage = (
        micro_trainer.curriculum.apply(
            MICRO_START_STEP
        )
    )
    micro_trainer.criterion.set_loss_weight_overrides(
        micro_trainer.curriculum_stage.loss_weight_overrides
    )

    before = evaluate_structure(
        micro_model,
        batch_gpu,
        target,
        label="pre_cr_micro_before",
        native=False,
    )

    first_loss = micro_trainer.train_step(
        batch
    )
    grad_gate = {
        "proposal_local_encoder": module_grad_norm(
            micro_model
            .spatial_proposal_generator
            .local_encoder
        ),
        "cr1_cross": module_grad_norm(
            micro_model.cr1.cross
        ),
        "cr2_cross": module_grad_norm(
            micro_model.cr2.cross
        ),
        "proposal_projection": module_grad_norm(
            micro_model
            .query_builder
            .proposal_proj
        ),
        "query_decoder": module_grad_norm(
            micro_model.query_decoder
        ),
    }

    print(
        "First micro-step loss:",
        first_loss["loss"],
    )
    print(
        "CUDA peak during first backward:",
        f"{torch.cuda.max_memory_allocated() / 1024**3:.2f} GiB",
    )
    print("Gradient norms:")
    for key, value in grad_gate.items():
        print(
            f"  {key:24s} {value:.6g}"
        )

    if (
        grad_gate["cr1_cross"] == 0
        or grad_gate["cr2_cross"] == 0
    ):
        raise RuntimeError(
            "Pre-CR proposal feedback is not "
            "reaching both co-reasoning blocks."
        )

    micro_history = [first_loss]
    for _ in range(MICRO_STEPS - 1):
        micro_history.append(
            micro_trainer.train_step(batch)
        )

    after = evaluate_structure(
        micro_model,
        batch_gpu,
        target,
        label=(
            f"pre_cr_micro_after_"
            f"{MICRO_STEPS}"
        ),
        native=False,
    )

    micro_result = pd.DataFrame(
        [before, after]
    )
    display(micro_result.T)
    micro_result.to_csv(
        RUN_DIR / "micro_screen.csv",
        index=False,
    )

    with (
        RUN_DIR / "micro_gradient_gate.json"
    ).open("w", encoding="utf-8") as handle:
        json.dump(
            grad_gate, handle, indent=2
        )

    del micro_trainer, micro_model
    torch.cuda.empty_cache()
    gc.collect()
else:
    print("Micro screen disabled.")


## 8. Full comparable staged run: step 30 → 85

This is the Notebook-25 result to compare against Notebook 24.

The model starts again from the untouched migrated step-30 seed. The current repository curriculum is used without inventing a new optimizer policy:

| Step range | Stage |
|---|---|
| 30–49 | temporal_dense |
| 50–69 | query_bootstrap |
| 70–79 | native_bootstrap |
| 80–84 | joint |

Snapshots are evaluated at steps **30 / 50 / 70 / 80 / 85**.

The pre-CR preview path remains active throughout. `spatial_dense` itself is not repeated because both architectures start from the same saved step-30 spatial state.


In [ ]:
snapshot_steps = {
    30, 50, 70, 80, 85
}
history = []
snapshots = []

early_model = None
early_trainer = None

if RUN_FULL_COMPARABLE:
    early_model = StirNet(cfg)
    early_model.load_state_dict(
        seed_state_cpu, strict=True
    )
    bind_precr_patch(
        early_model,
        inject_proposals=True,
    )

    early_trainer = Trainer(
        early_model,
        cfg,
        device=device,
        amp_dtype=AMP_DTYPE,
    )
    early_trainer.global_step = START_STEP
    early_trainer.curriculum_stage = (
        early_trainer.curriculum.apply(
            START_STEP
        )
    )
    early_trainer.criterion.set_loss_weight_overrides(
        early_trainer.curriculum_stage.loss_weight_overrides
    )

    row = evaluate_structure(
        early_model,
        batch_gpu,
        target,
        label="pre_cr_step30",
        native=False,
    )
    row["step"] = 30
    row["stage"] = curriculum_stage(
        cfg.curriculum, 30
    ).name
    snapshots.append(row)

    wall_start = time.perf_counter()

    while (
        early_trainer.global_step
        < FINAL_STEP
    ):
        step_before = (
            early_trainer.global_step
        )
        stage = curriculum_stage(
            cfg.curriculum,
            step_before,
        ).name

        t0 = time.perf_counter()
        losses = early_trainer.train_step(
            batch
        )
        torch.cuda.synchronize()
        dt = time.perf_counter() - t0

        record = {
            "step": early_trainer.global_step,
            "stage": stage,
            "seconds": dt,
            **losses,
        }
        history.append(record)

        if (
            early_trainer.global_step
            % LOG_EVERY
            == 0
            or early_trainer.global_step
            in snapshot_steps
        ):
            print(
                f"step "
                f"{early_trainer.global_step:3d}"
                f" | {stage:16s}"
                f" | loss "
                f"{losses['loss']:.5f}"
                f" | {dt:.1f}s"
            )

        if (
            early_trainer.global_step
            in snapshot_steps
        ):
            s = evaluate_structure(
                early_model,
                batch_gpu,
                target,
                label=(
                    f"pre_cr_step"
                    f"{early_trainer.global_step}"
                ),
                native=False,
            )
            s["step"] = (
                early_trainer.global_step
            )
            s["stage"] = curriculum_stage(
                cfg.curriculum,
                early_trainer.global_step,
            ).name
            snapshots.append(s)

    total_minutes = (
        time.perf_counter()
        - wall_start
    ) / 60.0
    print(
        "Full comparable run completed "
        f"in {total_minutes:.1f} min"
    )

    history_df = pd.DataFrame(history)
    snapshot_df = pd.DataFrame(
        snapshots
    )
    history_df.to_csv(
        RUN_DIR / "training_history.csv",
        index=False,
    )
    snapshot_df.to_csv(
        RUN_DIR / "snapshot_metrics.csv",
        index=False,
    )

    save_checkpoint(
        RUN_DIR / "checkpoint_final.pt",
        model=early_model,
        optimizer=early_trainer.optimizer,
        scaler=early_trainer.scaler,
        step=early_trainer.global_step,
        config=cfg,
        extra={
            "notebook": 25,
            "architecture": (
                "pre_coreasoning_"
                "spatial_proposals"
            ),
        },
    )

    display(snapshot_df.T)
else:
    print("Full comparable run disabled.")


## 9. Training curves

Loss is secondary; the structural snapshots above are the primary verdict. These plots are only to catch optimization instability.


In [ ]:
if (
    RUN_FULL_COMPARABLE
    and len(history_df)
):
    for metric in [
        "loss",
        "loss_coarse_dice",
        "loss_high_dice",
        "loss_center",
        "loss_proposal_center",
        "loss_internal_boundary",
    ]:
        if metric not in history_df.columns:
            continue
        plt.figure(figsize=(8, 3.5))
        plt.plot(
            history_df["step"],
            history_df[metric],
        )
        plt.xlabel("step")
        plt.ylabel(metric)
        plt.title(
            f"Notebook 25 — {metric}"
        )
        plt.grid(alpha=0.25)
        plt.show()


## 10. Final source-9 native masks

Native rendering is intentionally deferred to the final checkpoint because it is expensive at the full native lattice.

Each matched source-9 query is rendered **one at a time** to control GPU memory.


In [ ]:
final_metrics = None

if RUN_FULL_COMPARABLE:
    final_metrics = evaluate_structure(
        early_model,
        batch_gpu,
        target,
        label="pre_cr_step85_final",
        native=COMPUTE_NATIVE_AT_FINAL,
    )
    final_metrics["step"] = (
        early_trainer.global_step
    )
    final_metrics["stage"] = "joint"

    display(
        pd.DataFrame(
            [final_metrics]
        ).T
    )

    with (
        RUN_DIR
        / "summary_notebook25.json"
    ).open(
        "w", encoding="utf-8"
    ) as handle:
        json.dump(
            final_metrics,
            handle,
            indent=2,
            default=lambda x: (
                x.item()
                if isinstance(x, np.generic)
                else x
            ),
        )


## 11. Same-weight final routing ablation

This separates **what the trained weights learned** from **where proposal identity is routed**.

Using the final Notebook-25 weights:

1. `pre_cr_source9_read`: the low-VRAM architecture used during training; source-9 proposals exist before CR1 and read CR1/CR2 spatial states.
2. `pre_cr_no_read`: early proposals are generated but do not enter CR1/CR2.
3. `post_cr_current_same_weights`: identical weights loaded into the unmodified repository forward, so proposals are generated after CR2.

If `pre_cr_feedback` is clearly better than both same-weight alternatives, that is strong evidence that the *placement and participation* of proposal queries inside co-reasoning matters.


In [ ]:
routing_df = None

if RUN_FULL_COMPARABLE:
    final_state_cpu = {
        k: v.detach().cpu().clone()
        for k, v
        in early_model.state_dict().items()
    }

    early_model._notebook25_inject_proposals = True
    early_model._notebook25_spatial_feedback = False
    routing_rows = [
        evaluate_structure(
            early_model,
            batch_gpu,
            target,
            label=(
                "pre_cr_feedback_"
                "same_weights"
            ),
            native=False,
        )
    ]

    early_model._notebook25_inject_proposals = False
    routing_rows.append(
        evaluate_structure(
            early_model,
            batch_gpu,
            target,
            label=(
                "pre_cr_no_feedback_"
                "same_weights"
            ),
            native=False,
        )
    )
    early_model._notebook25_inject_proposals = True

    post_model = StirNet(cfg)
    post_model.load_state_dict(
        final_state_cpu, strict=True
    )
    post_model.to(device)

    routing_rows.append(
        evaluate_structure(
            post_model,
            batch_gpu,
            target,
            label=(
                "post_cr_current_"
                "same_weights"
            ),
            native=False,
        )
    )

    routing_df = pd.DataFrame(
        routing_rows
    )
    display(routing_df.T)
    routing_df.to_csv(
        RUN_DIR
        / "final_routing_ablation.csv",
        index=False,
    )

    del post_model
    torch.cuda.empty_cache()
    gc.collect()


## 12. Notebook-24 comparison

Notebook 24 is local and may not have the same summary filename, so this cell searches common run locations without assuming it exists.

The most important columns to compare are:

```text
proposal_recovery_0p5_dref
proposal_recovery_1p0_dref
source9_matched_gt_count
source9_coarse_dice_mean
source9_native_dice_mean
source9_pairwise_coarse_dice_mean
source9_pairwise_native_dice_mean
source9_center_error_mean_um
source9_internal_boundary_auc
foreground_hard_dice
forward_seconds
peak_memory_gib
```

### Preferred outcome for Notebook 25

The early-query architecture should not merely lower loss. It should show a structural change:

- source-9 proposal recovery stays high;
- **distinct matched GT count increases toward 9/9**;
- assigned coarse/native Dice increases;
- **pairwise sibling mask Dice drops materially**;
- query cosine similarity decreases through CR1/CR2;
- foreground/boundary quality is not seriously damaged.

A compute increase is expected because this notebook intentionally duplicates the decode path for a causal prototype.


In [ ]:
candidate_paths = []
for pattern in [
    "runs/stirnet/**/24*/**/*summary*.json",
    "runs/stirnet/**/24*/*summary*.json",
    "runs/stirnet/**/*notebook24*.json",
]:
    candidate_paths.extend(
        REPO_ROOT.glob(pattern)
    )

candidate_paths = sorted(
    set(candidate_paths)
)

print("Notebook-25 summary:")
print(
    RUN_DIR
    / "summary_notebook25.json"
)

if candidate_paths:
    print(
        "\nPossible Notebook-24 summaries:"
    )
    for path in candidate_paths:
        print(" -", path)
else:
    print(
        "\nNo Notebook-24 summary JSON was "
        "found automatically. That is expected "
        "while Notebook 24 is still running/"
        "not committed."
    )


## 13. Automatic Notebook-25 verdict

This verdict is intentionally relative to the Notebook-25 zero-step state and the final same-weight routing ablation. The decisive Notebook-24 vs Notebook-25 architecture choice should be made after both final summaries are available.


In [ ]:
if (
    RUN_FULL_COMPARABLE
    and final_metrics is not None
):
    zero_pre = zero_df[
        zero_df["label"]
        == "pre_cr_feedback_step30"
    ].iloc[0]

    print(
        "=== NOTEBOOK 25 STRUCTURAL VERDICT ==="
    )
    print(
        "source-9 matched GT:",
        int(
            zero_pre[
                "source9_matched_gt_count"
            ]
        ),
        "→",
        int(
            final_metrics[
                "source9_matched_gt_count"
            ]
        ),
        "/ 9",
    )
    print(
        "source-9 coarse Dice:",
        f"{zero_pre['source9_coarse_dice_mean']:.4f}",
        "→",
        f"{final_metrics['source9_coarse_dice_mean']:.4f}",
    )
    print(
        "pairwise coarse sibling Dice:",
        f"{zero_pre['source9_pairwise_coarse_dice_mean']:.4f}",
        "→",
        f"{final_metrics['source9_pairwise_coarse_dice_mean']:.4f}",
        "(lower is better)",
    )
    print(
        "proposal recovery @0.5 dref:",
        int(
            final_metrics[
                "proposal_recovery_0p5_dref"
            ]
        ),
        "/ 9",
    )
    print(
        "proposal recovery @1.0 dref:",
        int(
            final_metrics[
                "proposal_recovery_1p0_dref"
            ]
        ),
        "/ 9",
    )

    if (
        "source9_native_dice_mean"
        in final_metrics
    ):
        print(
            "final source-9 native Dice:",
            f"{final_metrics['source9_native_dice_mean']:.4f}",
        )
        print(
            "final pairwise native sibling Dice:",
            f"{final_metrics['source9_pairwise_native_dice_mean']:.4f}",
        )

    matched_gain = (
        final_metrics[
            "source9_matched_gt_count"
        ]
        - zero_pre[
            "source9_matched_gt_count"
        ]
    )
    coarse_gain = (
        final_metrics[
            "source9_coarse_dice_mean"
        ]
        - zero_pre[
            "source9_coarse_dice_mean"
        ]
    )
    collapse_drop = (
        zero_pre[
            "source9_pairwise_coarse_dice_mean"
        ]
        - final_metrics[
            "source9_pairwise_coarse_dice_mean"
        ]
    )

    if (
        final_metrics[
            "source9_matched_gt_count"
        ]
        >= 8
        and coarse_gain > 0
        and collapse_drop > 0.10
    ):
        print(
            "\nGREEN: early proposal-query placement "
            "produced the kind of decomposition "
            "change we were looking for."
        )
    elif (
        coarse_gain > 0
        or collapse_drop > 0.05
        or matched_gain > 0
    ):
        print(
            "\nYELLOW: the mechanism moved structural "
            "metrics, but Notebook 24 must decide "
            "whether the gain is large enough to "
            "justify the extra architectural complexity."
        )
    else:
        print(
            "\nRED: the early-query mechanism did not "
            "materially break source-9 mask symmetry "
            "in this run."
        )


## 14. Interpretation guide for Notebook 24 vs 25

### Case A — Notebook 25 wins strongly

If Notebook 25 gives:

- similar or better proposal-center recovery;
- more of the 9 GT cells matched distinctly;
- much lower sibling mask Dice;
- better assigned native Dice;

then the architectural conclusion is:

> **Proposal identities should exist before co-reasoning, so global spatial/temporal reasoning refines already-distinct cell hypotheses instead of reasoning first and creating cell identities afterward.**

A production refactor would then remove the duplicate preview decoder and introduce an efficient early high-resolution proposal stem.

### Case B — Notebook 24 wins

If Notebook 24 has equally good proposal recovery but better masks and lower collapse, keep the current post-CR proposal architecture. It would mean co-reasoning benefits from first improving spatial features before proposal extraction, while early proposal feedback adds little.

### Case C — Notebook 25 proposals are worse before CR, but feedback helps when proposals are good

That means the idea is not disproven. The bottleneck is the **preview proposal quality**. The next production design should create high-resolution proposals from an earlier dedicated spatial stem rather than from a full unreasoned decoder preview.

### Case D — proposals are good in both notebooks but masks still collapse

Then proposal timing is not the remaining bottleneck. Inspect:

- proposal-query embeddings after CR1/CR2;
- query-decoder attention diversity;
- dot-product mask feature expressivity;
- previous-mask feedback.

Do not respond by simply adding more proposal count or more spatial tokens.
